# 02 · Training and evaluation

Trains two Sentinel-1-only segmentation models (8 classes) on the global patches from notebook 01.
Training and validation use 2023–24; evaluation uses **2025–26**: the main test set covers 9 regions never seen
in training, and `test_temporal` covers the training regions, which only measures robustness over time.

| Model | Encoder | Why |
|---|---|---|
| **A · U-Net (scratch)** | 4 conv stages, 32→512 filters, random init | Baseline; no assumptions borrowed from optical images |
| **B · U-Net + ResNet50** | ImageNet-pretrained ResNet50 | Tests whether pretrained edge/texture filters transfer to SAR and help with limited data |

**Shared server:** the GPUs are used by others. The first cell pins one GPU (`CUDA_VISIBLE_DEVICES`) and
lets TensorFlow grow its memory instead of reserving all of it. Check `nvidia-smi` and pick the GPU with
more free memory. Set `SMOKE_TEST = True` first to check the whole notebook in a couple of minutes.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # change to "1" if GPU 0 is busy (check nvidia-smi)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import sys, json, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import tensorflow as tf
from tensorflow import keras

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
keras.mixed_precision.set_global_policy("mixed_float16")
keras.utils.set_random_seed(42)

from src import config as C, model as M, evaluation as E, patching as P

CMAP = ListedColormap(C.CLASS_COLORS)
AUTOTUNE = tf.data.AUTOTUNE
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
SMOKE_TEST = False     # True: 64 patches per split, 2 epochs - just to check that everything runs

BATCH = 16
EPOCHS = 2 if SMOKE_TEST else 60
PATIENCE = 10          # early stopping on validation mIoU
WEIGHTS_DIR = ROOT / "weights"
WEIGHTS_DIR.mkdir(exist_ok=True)
stats = M.load_stats()
stats

## 1. Data

Patches are loaded into RAM (float16 dB). The input pipeline clips and z-scores each channel with the
training-split statistics (**normalisation**). For training only, it applies **augmentation**:

| Augmentation | How | Why |
|---|---|---|
| Flips, 90° rotations | random horizontal flip, k × 90° rotation | land cover has no preferred orientation |
| Brightness change | the same random offset of ±1 dB added to VV and VH (VV−VH ratio unchanged) | mimics calibration and incidence-angle differences between scenes, orbits and satellites (Sentinel-1A in training vs. 1A + 1C in testing) |
| Speckle noise | Gaussian noise, σ = 0.3 dB, per pixel and channel | residual speckle varies with the number of scenes in a composite |

The brightness change works in dB, i.e. as a multiplicative change of backscatter power, which is how such
differences actually occur. Large or per-channel changes are avoided, as they would alter the physical meaning
(e.g. turn water-like backscatter into vegetation).

In [ ]:
def load_split(split):
    X = np.load(C.PATCH_DIR / f"X_{split}.npy")
    y = np.load(C.PATCH_DIR / f"y_{split}.npy")
    meta = pd.read_csv(C.PATCH_DIR / f"meta_{split}.csv")
    if SMOKE_TEST:
        return X[:64], y[:64], meta.iloc[:64]
    return X, y, meta

splits = [s for s in C.SPLITS if (C.PATCH_DIR / f"X_{s}.npy").exists()]
data = {s: load_split(s) for s in splits}
pd.DataFrame({s: {"patches": len(X), "areas": m.aoi.nunique(), "GB in RAM": round(X.nbytes / 1e9, 2)}
              for s, (X, y, m) in data.items()}).T

In [ ]:
BRIGHTNESS_DB = 1.0
NOISE_DB = 0.3


def augment_intensity(x, t):
    # brightness change and speckle-like noise in dB, before normalisation
    x = tf.cast(x, tf.float32)
    offset = tf.random.uniform([], -BRIGHTNESS_DB, BRIGHTNESS_DB)
    x = x + tf.stack([offset, offset, 0.0])
    x = x + tf.random.normal(tf.shape(x), stddev=NOISE_DB)
    return x, t


def augment(x, t):
    k = tf.random.uniform([], 0, 4, tf.int32)
    x, t = tf.image.rot90(x, k), tf.image.rot90(t, k)
    flip = tf.random.uniform([]) < 0.5
    x = tf.cond(flip, lambda: tf.image.flip_left_right(x), lambda: x)
    t = tf.cond(flip, lambda: tf.image.flip_left_right(t), lambda: t)
    return x, t


def make_ds(split, training=False):
    X, y, _ = data[split]

    def gen():
        idx = np.random.permutation(len(X)) if training else np.arange(len(X))
        for i in idx:
            yield X[i], y[i][..., None]

    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((C.PATCH_PX, C.PATCH_PX, 3), tf.float16),
        tf.TensorSpec((C.PATCH_PX, C.PATCH_PX, 1), tf.uint8)))
    if training:
        ds = ds.map(augment_intensity, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, t: (M.normalise(x, stats), tf.cast(t, tf.int32)), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)


x, t = next(iter(make_ds("train", training=True)))
print("batch:", x.shape, x.dtype, t.shape, t.dtype, "| channel means after normalisation:",
      np.round(x.numpy().mean((0, 1, 2)), 2))

## 2. Class imbalance

Some classes (e.g. snow/ice, flooded vegetation, bare) cover far fewer pixels than trees or crops. Cross-entropy is weighted with
median-frequency weights (clipped to [0.2, 10]) and combined with a soft Dice term, which optimises overlap
per class and is less sensitive to class frequency.

In [ ]:
dist = P.class_distribution("train")
class_weights = M.median_frequency_weights(dist.pixels)
pd.DataFrame({"pixels_%": dist.percent.round(2).values, "loss_weight": class_weights.round(2)}, index=C.CLASSES)

## 3. Training setup

* Loss: weighted cross-entropy + soft Dice, ignoring pixels with unreliable labels (index 255)
* Optimiser: AdamW (weight decay 1e-4) with cosine learning-rate decay
* Mixed precision (float16 compute, float32 output layer)
* Model selection: best **validation mIoU**, early stopping after 10 epochs without improvement

In [ ]:
class ValMIoU(keras.callbacks.Callback):
    """Exact validation mIoU / OA from the full confusion matrix, added to the epoch logs."""

    def __init__(self, ds):
        super().__init__()
        self.ds, self.miou, self.oa = ds, [], []

    def on_epoch_end(self, epoch, logs=None):
        cm = np.zeros((M.NUM_CLASSES, M.NUM_CLASSES), np.int64)
        for x, t in self.ds:
            pred = np.argmax(self.model(x, training=False), -1)
            cm += E.confusion(t.numpy()[..., 0], pred)
        s = E.scores(cm)
        self.miou.append(s["mIoU"]); self.oa.append(s["OA"])
        logs["val_miou"], logs["val_oa"] = s["mIoU"], s["OA"]
        print(f"  val_mIoU {s['mIoU']:.4f} | val_OA {s['OA']:.4f}")


def train(model, lr, name):
    steps = int(np.ceil(len(data["train"][0]) / BATCH))
    schedule = keras.optimizers.schedules.CosineDecay(lr, decay_steps=EPOCHS * steps, alpha=0.01)
    model.compile(optimizer=keras.optimizers.AdamW(schedule, weight_decay=1e-4),
                  loss=M.make_loss(class_weights))
    val_cb = ValMIoU(make_ds("val"))
    ckpt = WEIGHTS_DIR / f"{name}.keras"
    callbacks = [
        val_cb,
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_miou", mode="max", save_best_only=True),
        keras.callbacks.EarlyStopping(monitor="val_miou", mode="max", patience=PATIENCE),
        keras.callbacks.CSVLogger(WEIGHTS_DIR / f"{name}_log.csv"),
    ]
    t0 = time.time()
    hist = model.fit(make_ds("train", training=True), validation_data=make_ds("val"),
                     epochs=EPOCHS, callbacks=callbacks, verbose=2)
    history = pd.DataFrame(hist.history).assign(val_miou=val_cb.miou, val_oa=val_cb.oa)
    history.index = history.index + 1
    info = dict(params=model.count_params(), epochs=len(history),
                best_epoch=int(history.val_miou.idxmax()), minutes=round((time.time() - t0) / 60, 1))
    best = keras.models.load_model(ckpt, compile=False)
    return best, history, info

## 4. Model A · U-Net from scratch

In [ ]:
model_a = M.unet()
print(f"{model_a.name}: {model_a.count_params():,} parameters")
model_a, hist_a, info_a = train(model_a, lr=1e-3, name="unet_scratch")
info_a

## 5. Model B · U-Net with pretrained ResNet50 encoder

A lower learning rate protects the pretrained encoder weights during fine-tuning.

In [ ]:
model_b = M.resnet50_unet()
print(f"{model_b.name}: {model_b.count_params():,} parameters")
model_b, hist_b, info_b = train(model_b, lr=3e-4, name="unet_resnet50")
info_b

## 6. Training curves

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for h, label, color in [(hist_a, "A · U-Net scratch", "#2563eb"), (hist_b, "B · U-Net ResNet50", "#dc2626")]:
    ax[0].plot(h.index, h.loss, color=color, label=f"{label} train")
    ax[0].plot(h.index, h.val_loss, color=color, ls="--", label=f"{label} val")
    ax[1].plot(h.index, h.val_miou, color=color, label=label)
    ax[2].plot(h.index, h.val_oa, color=color, label=label)
for a, title in zip(ax, ["Loss (weighted CE + Dice)", "Validation mIoU", "Validation overall accuracy"]):
    a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout()
plt.savefig(WEIGHTS_DIR / "training_curves.png", dpi=150)
plt.show()

## 7. Test-set evaluation

Main test set: 9 regions never seen in training (Dubai, Mexico City, Dhaka, Kansas, Canterbury NZ, Gabon,
Pantanal, Atacama, Vatnajökull), all from **2025–26**, i.e. unseen places *and* future dates.

In [ ]:
def predict(model, X, batch=32):
    out = np.empty(X.shape[:3], np.uint8)
    for i in range(0, len(X), batch):
        out[i:i + batch] = np.argmax(model.predict_on_batch(M.normalise(X[i:i + batch], stats)), -1)
    return out

X_test, y_test, meta_test = data["test"]
results = {}
for name, model, info in [("A · U-Net scratch", model_a, info_a), ("B · U-Net ResNet50", model_b, info_b)]:
    pred = predict(model, X_test)
    cm = E.confusion(y_test, pred)
    results[name] = dict(pred=pred, cm=cm, info=info, **E.scores(cm))

comparison = pd.DataFrame({n: {"OA": r["OA"], "mIoU": r["mIoU"], "mean F1": r["mF1"],
                               "parameters (M)": r["info"]["params"] / 1e6, "epochs": r["info"]["epochs"],
                               "best epoch": r["info"]["best_epoch"], "train minutes": r["info"]["minutes"]}
                           for n, r in results.items()}).T
comparison.round(4)

### Unseen regions vs. known regions (2025–26)

`test_temporal` applies the models to the *training* regions in 2025–26. The gap to the main test shows how much
of the error comes from new places rather than from the passage of time (and from Sentinel-1C adding more scenes
per composite from 2025 on).

In [ ]:
temporal = {}
if "test_temporal" in data:
    X_tt, y_tt, meta_tt = data["test_temporal"]
    for name, model in [("A · U-Net scratch", model_a), ("B · U-Net ResNet50", model_b)]:
        s = E.scores(E.confusion(y_tt, predict(model, X_tt)))
        temporal[name] = {"OA": s["OA"], "mIoU": s["mIoU"], "mean F1": s["mF1"]}
    display(pd.concat({"test: unseen regions": comparison[["OA", "mIoU", "mean F1"]].astype(float),
                       "test_temporal: training regions": pd.DataFrame(temporal).T}, axis=1).round(4))
else:
    print("No test_temporal split found")

In [ ]:
per_class = pd.concat({n: r["per_class"][["IoU", "F1", "precision", "recall"]] for n, r in results.items()}, axis=1)
per_class.insert(0, "support_%", next(iter(results.values()))["per_class"]["support_%"])
per_class.round(3)

In [ ]:
# the model is selected on validation mIoU, never on the test set
val_best = {"A · U-Net scratch": hist_a.val_miou.max(), "B · U-Net ResNet50": hist_b.val_miou.max()}
best_name = max(val_best, key=val_best.get)
print("Selected on validation mIoU:", best_name, {k: round(v, 4) for k, v in val_best.items()})

cm = results[best_name]["cm"].astype(float)
with np.errstate(invalid="ignore"):
    cm_norm = cm / cm.sum(1, keepdims=True)

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(M.NUM_CLASSES), C.CLASSES, rotation=45, ha="right")
ax.set_yticks(range(M.NUM_CLASSES), C.CLASSES)
for i in range(M.NUM_CLASSES):
    for j in range(M.NUM_CLASSES):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=9)
ax.set_xlabel("predicted"); ax.set_ylabel("reference (Dynamic World)")
ax.set_title(f"Test confusion matrix, row-normalised · {best_name}")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(WEIGHTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

### Per region and per month

Per-region scores show how well the model transfers to each unseen landscape. Per-month scores
show seasonal effects, e.g. bare fields after harvest, flooding in the wet season or wet snow lowering
backscatter. Note the test regions span both hemispheres, so calendar months mix opposite seasons.

In [ ]:
def grouped(y, pred, keys):
    rows = {}
    for k in sorted(set(keys)):
        m = np.asarray(keys) == k
        s = E.scores(E.confusion(y[m], pred[m]))
        rows[k] = {"OA": s["OA"], "mIoU": s["mIoU"], "patches": int(m.sum())}
    return pd.DataFrame(rows).T

pred_best = results[best_name]["pred"]
display(grouped(y_test, pred_best, meta_test.aoi.values).round(3))

by_month = grouped(y_test, pred_best, pd.to_datetime(meta_test.month).dt.month.values)
ax = by_month[["OA", "mIoU"]].plot(marker="o", figsize=(9, 3.5), title=f"Test scores by calendar month · {best_name}")
ax.set_xlabel("month"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Example predictions (test regions)

In [ ]:
idx = np.random.default_rng(1).choice(len(X_test), 5, replace=False)
names = list(results)
fig, ax = plt.subplots(len(idx), 2 + len(names), figsize=(4 * (2 + len(names)), 4 * len(idx)))
for row, i in enumerate(idx):
    ax[row, 0].imshow(X_test[i, ..., 0].astype(np.float32), cmap="gray", vmin=-25, vmax=0)
    ax[row, 0].set_title(f"VV · {meta_test.aoi[i]} {meta_test.month[i]}")
    ax[row, 1].imshow(np.ma.masked_equal(y_test[i], C.IGNORE), cmap=CMAP, vmin=0, vmax=M.NUM_CLASSES - 1,
                      interpolation="nearest")
    ax[row, 1].set_title("Dynamic World")
    for j, n in enumerate(names):
        ax[row, 2 + j].imshow(results[n]["pred"][i], cmap=CMAP, vmin=0, vmax=M.NUM_CLASSES - 1,
                              interpolation="nearest")
        ax[row, 2 + j].set_title(n)
for a in ax.ravel():
    a.axis("off")
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in C.CLASS_COLORS], C.CLASSES,
           loc="lower center", ncol=M.NUM_CLASSES)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

## 9. Save the selected model

The model with the higher **validation** mIoU becomes `weights/best_model.keras`, used by the prototype.
Test scores are reported for both, but the test set plays no part in the choice.

In [ ]:
model_sel = model_a if best_name.startswith("A") else model_b
model_sel.save(WEIGHTS_DIR / "best_model.keras")

summary = {
    "selected_model": best_name,
    "val_mIoU": {k: float(v) for k, v in val_best.items()},
    "test": {n: {"OA": float(r["OA"]), "mIoU": float(r["mIoU"]), "mF1": float(r["mF1"]),
                 "per_class_IoU": r["per_class"]["IoU"].round(4).to_dict(),
                 "per_class_F1": r["per_class"]["F1"].round(4).to_dict(), **r["info"]}
             for n, r in results.items()},
    "test_temporal": {n: {k: float(v) for k, v in d.items()} for n, d in temporal.items()},
    "classes": C.CLASSES,
}
(WEIGHTS_DIR / "results.json").write_text(json.dumps(summary, indent=2))
print("Selected:", best_name, "-> weights/best_model.keras")